# Thesis Visuals v4 — XGBoost(GPU) vs DCC-GARCH + DM tests + LaTeX tables

**Put this file into your project root** (e.g. `D:\progr`) and **Run All**.

## Prerequisite (must run once before this notebook)
Run your pipeline so `outputs/` exists and prediction exports are created:

```powershell
D:\progr\.venv\Scripts\python.exe D:\progr\main.py
```

## What this notebook produces
It reads your existing outputs and generates **thesis-ready artifacts**:

- **Figures** → `outputs/figures_notebook_v4/`
  - RMSE sensitivity vs window (per dependency)
  - RMSE bar comparison (XGB vs DCC) (per dependency)
  - Example forecast overlays (y_true vs XGB vs DCC) for selected cases
- **Tables** → `outputs/tables_notebook_v4/`
  - LaTeX table: all DM tests (XGB vs DCC)
  - LaTeX table: wins summary (how many significant wins)
  - LaTeX table: RMSE comparison (XGB vs DCC)

## Notes
- Uses **matplotlib only** (no seaborn).
- Trains **XGBoost on GPU** (RTX 4060) with `device="cuda"`.
- DM sign convention here: **DM < 0 ⇒ XGB has smaller loss (better)**.

In [1]:
# --- Imports & paths (RUN FIRST) ---
from __future__ import annotations

import re
import math
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error
import xgboost as xgb

# Root: assume notebook placed in project root (D:\progr)
ROOT = Path(".").resolve()

OUTPUTS = ROOT / "outputs"
RESULTS_DIR = OUTPUTS / "results"
PRED_DIR = OUTPUTS / "predictions"

FIG_DIR = OUTPUTS / "figures_notebook_v4"
TAB_DIR = OUTPUTS / "tables_notebook_v4"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print("Notebook ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR, "exists?", RESULTS_DIR.exists())
print("PRED_DIR:", PRED_DIR, "exists?", PRED_DIR.exists())
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

Notebook ROOT: D:\progr
RESULTS_DIR: D:\progr\outputs\results exists? True
PRED_DIR: D:\progr\outputs\predictions exists? True
FIG_DIR: D:\progr\outputs\figures_notebook_v4
TAB_DIR: D:\progr\outputs\tables_notebook_v4


## 1) Load `metrics.csv` and `dm_tests.csv` (if present)

In [2]:
metrics_path = RESULTS_DIR / "metrics.csv"
dm_path = RESULTS_DIR / "dm_tests.csv"

assert metrics_path.exists(), f"Missing {metrics_path}. Run your pipeline first."
metrics = pd.read_csv(metrics_path)

# Normalize schema differences
rename_map = {}
if "w" in metrics.columns and "window" not in metrics.columns:
    rename_map["w"] = "window"
if "space" in metrics.columns and "target_space" not in metrics.columns:
    rename_map["space"] = "target_space"
if "n" in metrics.columns and "n_test" not in metrics.columns:
    rename_map["n"] = "n_test"
metrics = metrics.rename(columns=rename_map)

print("metrics columns:", list(metrics.columns))
print("metrics shape:", metrics.shape)
display(metrics.head(10))

if dm_path.exists():
    dm = pd.read_csv(dm_path)
    dm_rename = {}
    if "dm_stat" in dm.columns and "DM_stat" not in dm.columns:
        dm_rename["dm_stat"] = "DM_stat"
    if "space" not in dm.columns and "target_space" in dm.columns:
        dm_rename["target_space"] = "space"
    dm = dm.rename(columns=dm_rename)
    print("dm_tests columns:", list(dm.columns))
    print("dm_tests shape:", dm.shape)
    display(dm.head(10))
else:
    dm = None
    print("dm_tests.csv not found (OK).")

metrics columns: ['dependency', 'window', 'target_space', 'model', 'MAE', 'RMSE', 'R2', 'n_test']
metrics shape: (120, 8)


,dependency,window,target_space,model,MAE,RMSE,R2,n_test
0,corr_BTC-USD_^GSPC,14,fisher_z,AR1,0.088039,0.126297,0.868947,1255
1,corr_BTC-USD_^GSPC,14,fisher_z,Naive_Last,0.087919,0.127862,0.865678,1255
2,corr_BTC-USD_^GSPC,14,fisher_z,Ridge,0.132550,0.175257,0.747644,1255
3,corr_BTC-USD_^GSPC,14,fisher_z,RF,0.144552,0.186337,0.714725,1255
4,corr_BTC-USD_^GSPC,14,fisher_z,ElasticNet,0.320505,0.396812,-0.293702,1255
5,corr_BTC-USD_^GSPC,30,fisher_z,AR1,0.037334,0.058634,0.950826,1239
6,corr_BTC-USD_^GSPC,30,fisher_z,Naive_Last,0.036935,0.058815,0.950522,1239
7,corr_BTC-USD_^GSPC,30,fisher_z,Ridge,0.057792,0.083880,0.899365,1239
8,corr_BTC-USD_^GSPC,30,fisher_z,RF,0.067173,0.093549,0.874826,1239
9,corr_BTC-USD_^GSPC,30,fisher_z,ElasticNet,0.269685,0.323107,-0.493237,1239


dm_tests columns: ['dependency', 'window', 'space', 'model', 'benchmark', 'DM_stat', 'p_value', 'n']
dm_tests shape: (48, 8)


,dependency,window,space,model,benchmark,DM_stat,p_value,n
0,corr_BTC-USD_^GSPC,14,fisher_z,Ridge,Naive_Last,-11.880939,0.000000e+00,1255.0
1,corr_BTC-USD_^GSPC,14,fisher_z,Ridge,DCC_GARCH,16.354195,0.000000e+00,1255.0
2,corr_BTC-USD_^GSPC,30,fisher_z,Ridge,Naive_Last,-9.833022,0.000000e+00,1239.0
3,corr_BTC-USD_^GSPC,30,fisher_z,Ridge,DCC_GARCH,18.051013,0.000000e+00,1239.0
4,corr_BTC-USD_^GSPC,60,fisher_z,Ridge,Naive_Last,-8.205213,6.661338e-16,1209.0
5,corr_BTC-USD_^GSPC,60,fisher_z,Ridge,DCC_GARCH,22.059640,0.000000e+00,1209.0
6,corr_BTC-USD_^GSPC,90,fisher_z,Ridge,Naive_Last,-7.045262,3.141931e-12,1179.0
7,corr_BTC-USD_^GSPC,90,fisher_z,Ridge,DCC_GARCH,27.272930,0.000000e+00,1179.0
8,corr_BTC-USD_^IXIC,14,fisher_z,Ridge,Naive_Last,-11.722002,0.000000e+00,1255.0
9,corr_BTC-USD_^IXIC,14,fisher_z,Ridge,DCC_GARCH,15.910565,0.000000e+00,1255.0


## 2) Scan prediction exports and detect usable files

In [3]:
assert PRED_DIR.exists(), f"Missing {PRED_DIR}. Run pipeline to generate prediction exports."

pred_files = sorted(PRED_DIR.glob("*_w*_fisher_z_predictions.csv"))
pred_files += sorted([p for p in PRED_DIR.glob("*_predictions.csv") if "_w" not in p.name])

print("Found prediction files:", len(pred_files))
for p in pred_files[:25]:
    print(" -", p.name)

def parse_dep_window(name: str):
    m = re.match(r"^(?P<dep>.+?)_w(?P<w>\d+)_", name)
    if m:
        return m.group("dep"), int(m.group("w"))
    dep = name.replace("_predictions.csv", "")
    return dep, None

def load_pred_file(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for c in ["Date", "date", "timestamp", "Datetime", "dt"]:
        if c in df.columns:
            df = df.rename(columns={c:"Date"})
            break
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.sort_values("Date").reset_index(drop=True)
    return df

if pred_files:
    sample = load_pred_file(pred_files[0])
    print("Sample file:", pred_files[0].name)
    print("Columns:", list(sample.columns))
    display(sample.head(5))

Found prediction files: 30
 - corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w60_fisher_z_predictions.csv
 - corr_BTC-USD_^GSPC_w90_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w14_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w30_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w60_fisher_z_predictions.csv
 - corr_BTC-USD_^IXIC_w90_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w30_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w60_fisher_z_predictions.csv
 - corr_BTC-USD_ETH-USD_w90_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w14_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w30_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w60_fisher_z_predictions.csv
 - corr_BTC-USD_GLD_w90_fisher_z_predictions.csv
 - corr_BTC-USD_SLV_w14_fisher_z_predictions.csv
 - corr_BTC-USD_SLV_w30_fisher_z_predictions.csv
 - corr_BTC-USD_SLV_w60_fisher_z_predictions.csv
 - corr_BT

,Date,y_true,Naive_Last,AR1,ElasticNet,Ridge,RF,XGB_GPU,DCC_GARCH
0,2017-12-08,-0.273968,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-12-11,-0.212765,NaN,NaN,NaN,NaN,NaN,NaN,0.245741
2,2017-12-12,-0.159723,NaN,NaN,NaN,NaN,NaN,NaN,0.248020
3,2017-12-13,-0.095343,NaN,NaN,NaN,NaN,NaN,NaN,0.249574
4,2017-12-14,-0.008050,NaN,NaN,NaN,NaN,NaN,NaN,0.248651


## 3) Helpers: lag features, DM test, LaTeX export

In [4]:
def build_lag_features(y: np.ndarray, n_lags: int = 30):
    '''
    Turn a 1D series into supervised learning:
      X[t] = [y[t-1], y[t-2], ..., y[t-n_lags]]
      y_target[t] = y[t]
    Returns (X, y_target) aligned.
    '''
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n <= n_lags:
        return np.empty((0, n_lags), dtype=float), np.empty((0,), dtype=float)
    X = np.column_stack([y[i: n - n_lags + i] for i in range(n_lags)][::-1])
    y_target = y[n_lags:]
    return X, y_target

def dm_test(loss_a: np.ndarray, loss_b: np.ndarray):
    '''
    Simple Diebold–Mariano test for equal predictive accuracy (h=1).
    Normal approximation; two-sided p-value.

    Here we use d = loss_a - loss_b.
      DM < 0 => model A has smaller loss (better).
    '''
    d = np.asarray(loss_a, dtype=float) - np.asarray(loss_b, dtype=float)
    d = d[np.isfinite(d)]
    n = len(d)
    if n < 5:
        return np.nan, np.nan, n
    mean_d = d.mean()
    var_d = d.var(ddof=1)
    if var_d <= 0:
        return np.nan, np.nan, n
    dm_stat = mean_d / math.sqrt(var_d / n)

    from math import erf, sqrt
    def norm_cdf(x): return 0.5 * (1.0 + erf(x / sqrt(2.0)))
    p = 2.0 * (1.0 - norm_cdf(abs(dm_stat)))
    return float(dm_stat), float(p), int(n)

def df_to_latex_table(df: pd.DataFrame, path: Path, caption: str, label: str,
                      float_format="%.4f", index=False):
    latex = df.to_latex(index=index, escape=True, float_format=lambda x: float_format % x)
    wrapped = "\n".join([
        r"\begin{table}[H]",
        r"\centering",
        r"\small",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        latex,
        r"\end{table}",
        ""
    ])
    path.write_text(wrapped, encoding="utf-8")
    return path

## 4) XGBoost(GPU) vs DCC-GARCH — automatic loop (DM + RMSE)

In [5]:
@dataclass
class LoopConfig:
    n_lags: int = 30
    train_frac: float = 0.80
    n_estimators: int = 600
    max_depth: int = 4
    learning_rate: float = 0.05
    subsample: float = 0.9
    colsample_bytree: float = 0.9
    reg_lambda: float = 1.0
    min_child_weight: float = 1.0
    random_state: int = 42

cfg = LoopConfig()

rows = []
skipped = []

for path in pred_files:
    dep, w = parse_dep_window(path.name)
    df = load_pred_file(path)

    target_col = None
    for cand in ["y_true", "target", "actual", "y", "corr", "value"]:
        if cand in df.columns:
            target_col = cand
            break
    if target_col is None:
        skipped.append((path.name, "no target column"))
        continue

    if "DCC_GARCH" not in df.columns:
        skipped.append((path.name, "no DCC_GARCH column"))
        continue

    y = df[target_col].to_numpy(dtype=float)
    X, y_target = build_lag_features(y, n_lags=cfg.n_lags)
    if len(y_target) < 100:
        skipped.append((path.name, f"too few samples after lags ({len(y_target)})"))
        continue

    dcc = df["DCC_GARCH"].to_numpy(dtype=float)[cfg.n_lags:]
    n = len(y_target)
    n_train = int(cfg.train_frac * n)
    if n_train <= 0 or n_train >= n:
        skipped.append((path.name, f"bad split n={n}, n_train={n_train}"))
        continue

    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y_target[:n_train], y_target[n_train:]
    dcc_test = dcc[n_train:]

    m = np.isfinite(y_test) & np.isfinite(dcc_test)
    if m.sum() < 20:
        skipped.append((path.name, f"too few finite test rows ({m.sum()})"))
        continue
    X_test2 = X_test[m]
    y_test2 = y_test[m]
    dcc_test2 = dcc_test[m]

    xgb_model = xgb.XGBRegressor(
        n_estimators=cfg.n_estimators,
        max_depth=cfg.max_depth,
        learning_rate=cfg.learning_rate,
        subsample=cfg.subsample,
        colsample_bytree=cfg.colsample_bytree,
        reg_lambda=cfg.reg_lambda,
        min_child_weight=cfg.min_child_weight,
        random_state=cfg.random_state,
        tree_method="hist",
        device="cuda",
        predictor="gpu_predictor",
        objective="reg:squarederror",
        n_jobs=0,
    )
    xgb_model.fit(X_train, y_train)

    pred_xgb = xgb_model.predict(X_test2)

    loss_xgb = (y_test2 - pred_xgb) ** 2
    loss_dcc = (y_test2 - dcc_test2) ** 2

    dm_stat, p_value, n_dm = dm_test(loss_xgb, loss_dcc)

    rmse_xgb = float(np.sqrt(np.mean(loss_xgb)))
    rmse_dcc = float(np.sqrt(np.mean(loss_dcc)))
    mae_xgb = float(mean_absolute_error(y_test2, pred_xgb))
    mae_dcc = float(mean_absolute_error(y_test2, dcc_test2))

    rows.append({
        "dependency": dep,
        "window": w,
        "n_test": int(len(y_test2)),
        "DM_stat": dm_stat,
        "p_value": p_value,
        "RMSE_XGB": rmse_xgb,
        "RMSE_DCC": rmse_dcc,
        "MAE_XGB": mae_xgb,
        "MAE_DCC": mae_dcc,
        "file": path.name,
    })

dm_xgb_vs_dcc = pd.DataFrame(rows)
print("Rows computed:", len(dm_xgb_vs_dcc))
display(dm_xgb_vs_dcc.head(10))

if skipped:
    print("\nSkipped files:", len(skipped))
    display(pd.DataFrame(skipped, columns=["file","reason"]).head(20))

D:\progr\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:36:37] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
D:\progr\.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [22:36:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
D:\progr\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:36:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.


Rows computed: 24


,dependency,window,n_test,DM_stat,p_value,RMSE_XGB,RMSE_DCC,MAE_XGB,MAE_DCC,file
0,corr_BTC-USD_^GSPC,14,405,-11.801541,0.0,0.150846,0.313846,0.108791,0.247764,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv
1,corr_BTC-USD_^GSPC,30,402,-11.126840,0.0,0.069807,0.179033,0.045418,0.140491,corr_BTC-USD_^GSPC_w30_fisher_z_predictions.csv
2,corr_BTC-USD_^GSPC,60,396,-15.219096,0.0,0.038912,0.138831,0.024364,0.117403,corr_BTC-USD_^GSPC_w60_fisher_z_predictions.csv
3,corr_BTC-USD_^GSPC,90,390,-20.736872,0.0,0.026471,0.116303,0.015909,0.103087,corr_BTC-USD_^GSPC_w90_fisher_z_predictions.csv
4,corr_BTC-USD_^IXIC,14,405,-11.243049,0.0,0.152714,0.313793,0.106007,0.246290,corr_BTC-USD_^IXIC_w14_fisher_z_predictions.csv
5,corr_BTC-USD_^IXIC,30,402,-11.927412,0.0,0.068211,0.170093,0.045997,0.135938,corr_BTC-USD_^IXIC_w30_fisher_z_predictions.csv
6,corr_BTC-USD_^IXIC,60,396,-16.128564,0.0,0.037166,0.120879,0.023620,0.104239,corr_BTC-USD_^IXIC_w60_fisher_z_predictions.csv
7,corr_BTC-USD_^IXIC,90,390,-19.893886,0.0,0.023885,0.101095,0.015403,0.088738,corr_BTC-USD_^IXIC_w90_fisher_z_predictions.csv
8,corr_BTC-USD_ETH-USD,14,405,-11.885754,0.0,0.149677,0.295663,0.107591,0.239154,corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv
9,corr_BTC-USD_ETH-USD,30,402,-13.807434,0.0,0.073548,0.178799,0.050099,0.147215,corr_BTC-USD_ETH-USD_w30_fisher_z_predictions.csv



Skipped files: 6


,file,reason
0,corr_BTC-USD_^GSPC_predictions.csv,no DCC_GARCH column
1,corr_BTC-USD_^IXIC_predictions.csv,no DCC_GARCH column
2,corr_BTC-USD_ETH-USD_predictions.csv,no DCC_GARCH column
3,corr_BTC-USD_GLD_predictions.csv,no DCC_GARCH column
4,corr_BTC-USD_SLV_predictions.csv,no DCC_GARCH column
5,corr_BTC-USD_UUP_predictions.csv,no DCC_GARCH column


## 5) Save CSV + LaTeX tables

In [6]:
assert len(dm_xgb_vs_dcc) > 0, "No rows computed. Ensure prediction files contain DCC_GARCH and y_true."

out_csv = RESULTS_DIR / "dm_xgb_vs_dcc.csv"
dm_xgb_vs_dcc.sort_values(["dependency","window"]).to_csv(out_csv, index=False)
print("Saved:", out_csv)

dm_tbl = dm_xgb_vs_dcc.sort_values(["dependency","window"])[
    ["dependency","window","n_test","DM_stat","p_value","RMSE_XGB","RMSE_DCC"]
].copy()

tex_main = TAB_DIR / "dm_xgb_vs_dcc_table.tex"
df_to_latex_table(
    dm_tbl,
    tex_main,
    caption="Diebold--Mariano test comparing XGBoost (GPU) vs DCC-GARCH forecasts (squared-error loss). Negative DM indicates XGBoost is better.",
    label="tab:dm_xgb_vs_dcc",
    float_format="%.4f",
    index=False
)
print("Saved LaTeX:", tex_main)

sig = dm_xgb_vs_dcc[np.isfinite(dm_xgb_vs_dcc["p_value"]) & (dm_xgb_vs_dcc["p_value"] < 0.05)].copy()
sig["winner"] = np.where(sig["DM_stat"] < 0, "XGB_GPU", "DCC_GARCH")
wins = sig.groupby("winner").size().reset_index(name="count")

for who in ["XGB_GPU","DCC_GARCH"]:
    if who not in set(wins["winner"]):
        wins = pd.concat([wins, pd.DataFrame([{"winner":who,"count":0}])], ignore_index=True)

wins = wins.sort_values("winner").reset_index(drop=True)
display(wins)

tex_wins = TAB_DIR / "dm_wins_summary.tex"
df_to_latex_table(
    wins,
    tex_wins,
    caption="Number of significant wins (p<0.05) in DM tests across all dependencies and windows.",
    label="tab:dm_wins",
    float_format="%.0f",
    index=False
)
print("Saved LaTeX:", tex_wins)

rmse_tbl = dm_xgb_vs_dcc.sort_values(["dependency","window"])[
    ["dependency","window","RMSE_XGB","RMSE_DCC","MAE_XGB","MAE_DCC","n_test"]
].copy()
rmse_tbl["RMSE_impr_%"] = 100.0 * (rmse_tbl["RMSE_DCC"] - rmse_tbl["RMSE_XGB"]) / rmse_tbl["RMSE_DCC"]

tex_rmse = TAB_DIR / "rmse_xgb_vs_dcc_table.tex"
df_to_latex_table(
    rmse_tbl,
    tex_rmse,
    caption="Out-of-sample error comparison between XGBoost (GPU) and DCC-GARCH. Improvement is relative RMSE reduction.",
    label="tab:rmse_xgb_vs_dcc",
    float_format="%.4f",
    index=False
)
print("Saved LaTeX:", tex_rmse)

Saved: D:\progr\outputs\results\dm_xgb_vs_dcc.csv
Saved LaTeX: D:\progr\outputs\tables_notebook_v4\dm_xgb_vs_dcc_table.tex


,winner,count
0,DCC_GARCH,0
1,XGB_GPU,24


Saved LaTeX: D:\progr\outputs\tables_notebook_v4\dm_wins_summary.tex
Saved LaTeX: D:\progr\outputs\tables_notebook_v4\rmse_xgb_vs_dcc_table.tex


## 6) Figures: RMSE sensitivity vs window (per dependency)

In [7]:
for dep, g in dm_xgb_vs_dcc.groupby("dependency"):
    g = g.sort_values("window")
    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(g["window"], g["RMSE_XGB"], marker="o", label="XGB_GPU")
    ax.plot(g["window"], g["RMSE_DCC"], marker="o", label="DCC_GARCH")
    ax.set_title(f"RMSE sensitivity vs window — {dep}")
    ax.set_xlabel("Rolling window (days)")
    ax.set_ylabel("RMSE (Fisher-z target)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = FIG_DIR / f"rmse_sensitivity_{dep}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    print("Saved:", out.name)

Saved: rmse_sensitivity_corr_BTC-USD_ETH-USD.png
Saved: rmse_sensitivity_corr_BTC-USD_GLD.png
Saved: rmse_sensitivity_corr_BTC-USD_SLV.png
Saved: rmse_sensitivity_corr_BTC-USD_UUP.png
Saved: rmse_sensitivity_corr_BTC-USD_^GSPC.png
Saved: rmse_sensitivity_corr_BTC-USD_^IXIC.png


## 7) Figures: RMSE bar comparison (per dependency)

In [8]:
for dep, g in dm_xgb_vs_dcc.groupby("dependency"):
    g = g.sort_values("window")
    x = np.arange(len(g))
    width = 0.38
    fig, ax = plt.subplots(figsize=(7,4))
    ax.bar(x - width/2, g["RMSE_XGB"], width, label="XGB_GPU")
    ax.bar(x + width/2, g["RMSE_DCC"], width, label="DCC_GARCH")
    ax.set_title(f"RMSE comparison — {dep}")
    ax.set_xlabel("Rolling window (days)")
    ax.set_ylabel("RMSE")
    ax.set_xticks(x, [str(int(w)) for w in g["window"]])
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    out = FIG_DIR / f"rmse_bar_{dep}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    print("Saved:", out.name)

Saved: rmse_bar_corr_BTC-USD_ETH-USD.png
Saved: rmse_bar_corr_BTC-USD_GLD.png
Saved: rmse_bar_corr_BTC-USD_SLV.png
Saved: rmse_bar_corr_BTC-USD_UUP.png
Saved: rmse_bar_corr_BTC-USD_^GSPC.png
Saved: rmse_bar_corr_BTC-USD_^IXIC.png


## 8) Example forecast overlays (y_true vs XGB vs DCC) for selected cases

In [9]:
examples = []
for dep, g in dm_xgb_vs_dcc.groupby("dependency"):
    g = g.sort_values("window")
    examples.append(g.iloc[0])
    examples.append(g.iloc[-1])

examples_df = pd.DataFrame(examples).drop_duplicates(subset=["dependency","window"]).reset_index(drop=True)
display(examples_df[["dependency","window","file","DM_stat","p_value","RMSE_XGB","RMSE_DCC"]])

def overlay_plot(file_name: str, dep: str, window: int):
    path = PRED_DIR / file_name
    df = load_pred_file(path)
    if "y_true" not in df.columns or "DCC_GARCH" not in df.columns:
        return None

    y = df["y_true"].to_numpy(dtype=float)
    X, y_target = build_lag_features(y, n_lags=cfg.n_lags)
    if len(y_target) < 200:
        return None

    n = len(y_target)
    n_train = int(cfg.train_frac * n)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y_target[:n_train], y_target[n_train:]
    dcc = df["DCC_GARCH"].to_numpy(dtype=float)[cfg.n_lags:]
    dcc_test = dcc[n_train:]

    if "Date" in df.columns:
        dates = df["Date"].to_numpy()[cfg.n_lags:]
        dates_test = dates[n_train:]
    else:
        dates_test = np.arange(len(y_test))

    m = np.isfinite(y_test) & np.isfinite(dcc_test)
    if m.sum() < 50:
        return None

    X_test2, y_test2, dcc_test2, dates_test2 = X_test[m], y_test[m], dcc_test[m], dates_test[m]

    xgb_model = xgb.XGBRegressor(
        n_estimators=cfg.n_estimators,
        max_depth=cfg.max_depth,
        learning_rate=cfg.learning_rate,
        subsample=cfg.subsample,
        colsample_bytree=cfg.colsample_bytree,
        reg_lambda=cfg.reg_lambda,
        min_child_weight=cfg.min_child_weight,
        random_state=cfg.random_state,
        tree_method="hist",
        device="cuda",
        predictor="gpu_predictor",
        objective="reg:squarederror",
        n_jobs=0,
    )
    xgb_model.fit(X_train, y_train)
    pred_xgb = xgb_model.predict(X_test2)

    fig, ax = plt.subplots(figsize=(9,4))
    ax.plot(dates_test2, y_test2, label="y_true", linewidth=1.5)
    ax.plot(dates_test2, pred_xgb, label="XGB_GPU (retrained)", linewidth=1.2)
    ax.plot(dates_test2, dcc_test2, label="DCC_GARCH", linewidth=1.2)
    ax.set_title(f"Forecast overlay — {dep} — w={window}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Target (Fisher-z)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = FIG_DIR / f"overlay_{dep}_w{window}.png"
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    print("Saved:", out.name)
    return out

for _, row in examples_df.iterrows():
    overlay_plot(row["file"], row["dependency"], int(row["window"]))

,dependency,window,file,DM_stat,p_value,RMSE_XGB,RMSE_DCC
0,corr_BTC-USD_ETH-USD,14,corr_BTC-USD_ETH-USD_w14_fisher_z_predictions.csv,-11.885754,0.0,0.149677,0.295663
1,corr_BTC-USD_ETH-USD,90,corr_BTC-USD_ETH-USD_w90_fisher_z_predictions.csv,-13.585710,0.0,0.024252,0.137119
2,corr_BTC-USD_GLD,14,corr_BTC-USD_GLD_w14_fisher_z_predictions.csv,-11.727374,0.0,0.145329,0.314185
3,corr_BTC-USD_GLD,90,corr_BTC-USD_GLD_w90_fisher_z_predictions.csv,-14.969872,0.0,0.026373,0.078364
4,corr_BTC-USD_SLV,14,corr_BTC-USD_SLV_w14_fisher_z_predictions.csv,-11.933191,0.0,0.130975,0.315928
5,corr_BTC-USD_SLV,90,corr_BTC-USD_SLV_w90_fisher_z_predictions.csv,-16.951832,0.0,0.022290,0.078356
6,corr_BTC-USD_UUP,14,corr_BTC-USD_UUP_w14_fisher_z_predictions.csv,-12.199984,0.0,0.139788,0.349302
7,corr_BTC-USD_UUP,90,corr_BTC-USD_UUP_w90_fisher_z_predictions.csv,-14.387389,0.0,0.031762,0.104711
8,corr_BTC-USD_^GSPC,14,corr_BTC-USD_^GSPC_w14_fisher_z_predictions.csv,-11.801541,0.0,0.150846,0.313846
9,corr_BTC-USD_^GSPC,90,corr_BTC-USD_^GSPC_w90_fisher_z_predictions.csv,-20.736872,0.0,0.026471,0.116303


D:\progr\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:37:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Saved: overlay_corr_BTC-USD_ETH-USD_w14.png


D:\progr\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:37:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Saved: overlay_corr_BTC-USD_ETH-USD_w90.png


...

Saved: overlay_corr_BTC-USD_^GSPC_w90.png


D:\progr\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:37:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Saved: overlay_corr_BTC-USD_^IXIC_w14.png


D:\progr\.venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:37:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Saved: overlay_corr_BTC-USD_^IXIC_w90.png


## 9) Summary / where to insert into LaTeX

In [10]:
print("=== GENERATED ARTIFACTS ===")
print("Figures folder:", FIG_DIR)
print("Tables folder :", TAB_DIR)

print("\nKey LaTeX tables:")
for p in [TAB_DIR/"dm_xgb_vs_dcc_table.tex", TAB_DIR/"dm_wins_summary.tex", TAB_DIR/"rmse_xgb_vs_dcc_table.tex"]:
    print(" -", p.name, "exists?", p.exists())

print("\nHow to use in your thesis (LaTeX):")
print(r" - Put figures into your thesis with \includegraphics{outputs/figures_notebook_v4/<name>.png}")
print(r" - Insert tables with \input{outputs/tables_notebook_v4/<name>.tex}")

=== GENERATED ARTIFACTS ===
Figures folder: D:\progr\outputs\figures_notebook_v4
Tables folder : D:\progr\outputs\tables_notebook_v4

Key LaTeX tables:
 - dm_xgb_vs_dcc_table.tex exists? True
 - dm_wins_summary.tex exists? True
 - rmse_xgb_vs_dcc_table.tex exists? True

How to use in your thesis (LaTeX):
 - Put figures into your thesis with \includegraphics{outputs/figures_notebook_v4/<name>.png}
 - Insert tables with \input{outputs/tables_notebook_v4/<name>.tex}
